# NOTEBOOK 2: LIMPIEZA, ESTANDARIZACIÓN Y ETL

---

## Proyecto: Arquitectura de BI y Big Data para Análisis del Turismo Académico en Medellín

**Objetivo del Notebook:** Limpiar, estandarizar y consolidar los datos de las tres universidades en un único dataset de alta calidad listo para análisis.

---

### Contenido:
1. Carga de datos crudos
2. Funciones de limpieza reutilizables
3. Eliminación de valores nulos y duplicados
4. Estandarización de nomenclaturas
5. Creación de variables derivadas
6. Validación de calidad post-limpieza
7. Consolidación y exportación

---
## 1. CONFIGURACIÓN E IMPORTACIÓN

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
from datetime import datetime

# Configuración
BASE_DIR = Path(os.getcwd())
DATA_RAW_DIR = BASE_DIR / 'data' / 'raw'
DATA_PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
REPORTS_DIR = BASE_DIR / 'outputs' / 'reportes'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuración completada")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2. FUNCIONES DE LIMPIEZA REUTILIZABLES

In [ ]:
def limpiar_espacios(df, columnas):
    """
    Elimina espacios en blanco adicionales en columnas de texto.
    """
    df_clean = df.copy()
    for col in columnas:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip()
    return df_clean


def eliminar_filas_vacias(df):
    """
    Elimina filas completamente vacías.
    """
    antes = len(df)
    df_clean = df.dropna(how='all')
    despues = len(df_clean)
    eliminadas = antes - despues
    
    print(f"  - Filas eliminadas (vacías): {eliminadas:,}")
    print(f"  - Filas restantes: {despues:,}")
    
    return df_clean


def estandarizar_paises(df, columna='PAIS_EXTRANJERO'):
    """
    Estandariza nombres de países a nomenclatura oficial.
    """
    diccionario_paises = {
        'ESTADOS UNIDOS DE AMÉRICA': 'Estados Unidos',
        'ESTADOS UNIDOS': 'Estados Unidos',
        'USA': 'Estados Unidos',
        'US': 'Estados Unidos',
        'MÉXICO': 'México',
        'MEXICO': 'México',
        'ESPAÑA': 'España',
        'BRASIL': 'Brasil',
        'ITALY': 'Italia',
        'ITALIA': 'Italia',
        'GRECIA': 'Grecia',
        'GREECE': 'Grecia',
        'TURQUÍA': 'Turquía',
        'TURKEY': 'Turquía',
        'RUMANÍA': 'Rumania',
        'ROMANIA': 'Rumania',
        'VENEZUELA': 'Venezuela'
    }
    
    df_clean = df.copy()
    if columna in df_clean.columns:
        df_clean[columna] = df_clean[columna].str.upper().str.strip()
        df_clean[columna] = df_clean[columna].replace(diccionario_paises)
        
        paises_unicos_antes = df[columna].nunique()
        paises_unicos_despues = df_clean[columna].nunique()
        print(f"  - Países únicos antes: {paises_unicos_antes}")
        print(f"  - Países únicos después: {paises_unicos_despues}")
    
    return df_clean


def convertir_tipos_datos(df):
    """
    Convierte columnas a los tipos de datos apropiados.
    """
    df_clean = df.copy()
    
    # Convertir columnas numéricas
    columnas_numericas = ['AÑO', 'SEMESTRE', 'NUM_DIAS_MOVILIDAD', 
                          'VALOR_FINANCIACION_NACIONAL', 'VALOR_FINANCIACION_INTERNAC',
                          'ID_PAIS_EXTRANJERO', 'ID_TIPO_MOV_EST_EXTRANJ']
    
    for col in columnas_numericas:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    return df_clean


def crear_variables_derivadas(df):
    """
    Crea nuevas variables útiles para el análisis.
    """
    df_enhanced = df.copy()
    
    # 1. Periodo completo (Año-Semestre)
    if 'AÑO' in df_enhanced.columns and 'SEMESTRE' in df_enhanced.columns:
        df_enhanced['PERIODO'] = df_enhanced['AÑO'].astype(str) + '-' + df_enhanced['SEMESTRE'].astype(str)
    
    # 2. Nombre completo del estudiante
    if all(col in df_enhanced.columns for col in ['PRIMER_NOMBRE', 'PRIMER_APELLIDO']):
        df_enhanced['NOMBRE_COMPLETO'] = (
            df_enhanced['PRIMER_NOMBRE'].fillna('') + ' ' + 
            df_enhanced['SEGUNDO_NOMBRE'].fillna('') + ' ' + 
            df_enhanced['PRIMER_APELLIDO'].fillna('') + ' ' + 
            df_enhanced['SEGUNDO_APELLIDO'].fillna('')
        ).str.strip().str.replace(r'\s+', ' ', regex=True)
    
    # 3. Categoría de duración de movilidad
    if 'NUM_DIAS_MOVILIDAD' in df_enhanced.columns:
        def categorizar_duracion(dias):
            if pd.isna(dias):
                return 'No especificado'
            elif dias <= 7:
                return 'Corta (≤7 días)'
            elif dias <= 30:
                return 'Media (8-30 días)'
            elif dias <= 90:
                return 'Larga (31-90 días)'
            else:
                return 'Muy larga (>90 días)'
        
        df_enhanced['CATEGORIA_DURACION'] = df_enhanced['NUM_DIAS_MOVILIDAD'].apply(categorizar_duracion)
    
    # 4. Financiación total
    if 'VALOR_FINANCIACION_NACIONAL' in df_enhanced.columns and 'VALOR_FINANCIACION_INTERNAC' in df_enhanced.columns:
        df_enhanced['FINANCIACION_TOTAL'] = (
            df_enhanced['VALOR_FINANCIACION_NACIONAL'].fillna(0) + 
            df_enhanced['VALOR_FINANCIACION_INTERNAC'].fillna(0)
        )
    
    # 5. Tiene convenio (booleano)
    if 'MOVILIDAD_POR_CONVENIO' in df_enhanced.columns:
        df_enhanced['TIENE_CONVENIO'] = df_enhanced['MOVILIDAD_POR_CONVENIO'].isin(['S', 'SI', 'Sí', 'Y', 'Yes'])
    
    print(f"  - Variables derivadas creadas: {['PERIODO', 'NOMBRE_COMPLETO', 'CATEGORIA_DURACION', 'FINANCIACION_TOTAL', 'TIENE_CONVENIO']}")
    
    return df_enhanced


print("✓ Funciones de limpieza definidas")

---
## 3. CARGA Y LIMPIEZA DE DATOS - IUSH

In [ ]:
print("="*80)
print("PROCESAMIENTO ETL - IUSH")
print("="*80)

# Cargar datos
df_iush_raw = pd.read_csv(
    DATA_RAW_DIR / 'iush.csv',
    na_values=['#N/A', 'N/A', 'NA', '', ' '],
    keep_default_na=True
)

print(f"\n1. DATOS CRUDOS CARGADOS")
print(f"  - Registros: {len(df_iush_raw):,}")

# Paso 1: Eliminar filas vacías
print(f"\n2. ELIMINANDO FILAS VACÍAS")
df_iush_clean = eliminar_filas_vacias(df_iush_raw)

# Paso 2: Limpiar espacios en columnas de texto
print(f"\n3. LIMPIANDO ESPACIOS EN TEXTO")
columnas_texto = ['PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO',
                  'PAIS_EXTRANJERO', 'INSTITUCION_EXTRANJERA', 'TIPO_MOV_EST_EXTRANJ']
df_iush_clean = limpiar_espacios(df_iush_clean, columnas_texto)
print(f"  - Espacios eliminados en {len(columnas_texto)} columnas")

# Paso 3: Estandarizar nombres de países
print(f"\n4. ESTANDARIZANDO NOMBRES DE PAÍSES")
df_iush_clean = estandarizar_paises(df_iush_clean)

# Paso 4: Convertir tipos de datos
print(f"\n5. CONVIRTIENDO TIPOS DE DATOS")
df_iush_clean = convertir_tipos_datos(df_iush_clean)
print(f"  - Tipos de datos convertidos correctamente")

# Paso 5: Eliminar duplicados
print(f"\n6. ELIMINANDO DUPLICADOS")
antes_duplicados = len(df_iush_clean)
df_iush_clean = df_iush_clean.drop_duplicates()
duplicados_eliminados = antes_duplicados - len(df_iush_clean)
print(f"  - Duplicados eliminados: {duplicados_eliminados}")

# Paso 6: Crear variables derivadas
print(f"\n7. CREANDO VARIABLES DERIVADAS")
df_iush_clean = crear_variables_derivadas(df_iush_clean)

# Agregar identificador de universidad
df_iush_clean['UNIVERSIDAD'] = 'IUSH'

print(f"\n" + "="*80)
print(f"RESUMEN LIMPIEZA - IUSH")
print(f"="*80)
print(f"Registros iniciales: {len(df_iush_raw):,}")
print(f"Registros finales: {len(df_iush_clean):,}")
print(f"Reducción: {len(df_iush_raw) - len(df_iush_clean):,} ({((len(df_iush_raw) - len(df_iush_clean))/len(df_iush_raw)*100):.2f}%)")

---
## 4. VALIDACIÓN DE CALIDAD POST-LIMPIEZA

In [ ]:
print("\n" + "="*80)
print("VALIDACIÓN DE CALIDAD DE DATOS")
print("="*80)

# 1. Estadísticas de completitud
print("\n1. COMPLETITUD DE DATOS")
print("-" * 80)

completitud = pd.DataFrame({
    'Total_Registros': len(df_iush_clean),
    'Valores_Nulos': df_iush_clean.isnull().sum(),
    'Valores_Completos': len(df_iush_clean) - df_iush_clean.isnull().sum(),
    'Porcentaje_Completitud': ((len(df_iush_clean) - df_iush_clean.isnull().sum()) / len(df_iush_clean) * 100).round(2)
}).sort_values('Porcentaje_Completitud')

print(completitud.head(10))

# 2. Verificación de rangos válidos
print("\n2. VALIDACIÓN DE RANGOS")
print("-" * 80)

validaciones = []

# Años válidos (2020-2025)
if 'AÑO' in df_iush_clean.columns:
    años_validos = df_iush_clean['AÑO'].between(2020, 2025).sum()
    validaciones.append(f"✓ Años válidos (2020-2025): {años_validos}/{len(df_iush_clean)}")

# Semestres válidos (1-2)
if 'SEMESTRE' in df_iush_clean.columns:
    semestres_validos = df_iush_clean['SEMESTRE'].isin([1, 2]).sum()
    validaciones.append(f"✓ Semestres válidos (1-2): {semestres_validos}/{len(df_iush_clean)}")

# Días de movilidad positivos
if 'NUM_DIAS_MOVILIDAD' in df_iush_clean.columns:
    dias_positivos = (df_iush_clean['NUM_DIAS_MOVILIDAD'] > 0).sum()
    validaciones.append(f"✓ Días de movilidad positivos: {dias_positivos}/{len(df_iush_clean)}")

for val in validaciones:
    print(val)

# 3. Distribución de categorías
print("\n3. DISTRIBUCIÓN POR CATEGORÍAS")
print("-" * 80)

if 'PAIS_EXTRANJERO' in df_iush_clean.columns:
    print(f"\nPaíses únicos: {df_iush_clean['PAIS_EXTRANJERO'].nunique()}")
    print(f"Top 5 países:")
    print(df_iush_clean['PAIS_EXTRANJERO'].value_counts().head())

if 'TIPO_MOV_EST_EXTRANJ' in df_iush_clean.columns:
    print(f"\nTipos de movilidad:")
    print(df_iush_clean['TIPO_MOV_EST_EXTRANJ'].value_counts())

if 'CATEGORIA_DURACION' in df_iush_clean.columns:
    print(f"\nCategorías de duración:")
    print(df_iush_clean['CATEGORIA_DURACION'].value_counts())

---
## 5. CONSOLIDACIÓN DE DATOS (PREPARADO PARA 3 UNIVERSIDADES)

In [ ]:
# Por ahora solo tenemos IUSH, pero el código está preparado para consolidar
print("\n" + "="*80)
print("CONSOLIDACIÓN DE DATOS")
print("="*80)

# Lista de DataFrames a consolidar
dataframes = [df_iush_clean]
nombres_unis = ['IUSH']

# Cuando tengamos datos de UdeA y UNAC, agregaremos:
# if df_udea_clean is not None:
#     dataframes.append(df_udea_clean)
#     nombres_unis.append('Universidad de Antioquia')
# if df_unac_clean is not None:
#     dataframes.append(df_unac_clean)
#     nombres_unis.append('UNAC')

# Consolidar todos los datos
df_consolidado = pd.concat(dataframes, ignore_index=True)

print(f"\nUniversidades consolidadas: {', '.join(nombres_unis)}")
print(f"Total de registros: {len(df_consolidado):,}")
print(f"\nDistribución por universidad:")
print(df_consolidado['UNIVERSIDAD'].value_counts())

---
## 6. EXPORTACIÓN DE DATOS LIMPIOS

In [ ]:
# Exportar a CSV
archivo_salida = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.csv'
df_consolidado.to_csv(archivo_salida, index=False, encoding='utf-8')

print(f"\n✓ Datos consolidados exportados a: {archivo_salida}")
print(f"  - Registros: {len(df_consolidado):,}")
print(f"  - Columnas: {len(df_consolidado.columns)}")
print(f"  - Tamaño: {archivo_salida.stat().st_size / 1024:.2f} KB")

# Exportar también en formato Excel para revisión manual
archivo_excel = DATA_PROCESSED_DIR / 'datos_consolidados_limpios.xlsx'
df_consolidado.to_excel(archivo_excel, index=False, engine='openpyxl')

print(f"\n✓ Datos también exportados en Excel: {archivo_excel}")

---
## 7. REPORTE DE CALIDAD DE DATOS

In [ ]:
# Generar reporte de calidad
reporte_calidad = f"""
{'='*80}
REPORTE DE CALIDAD DE DATOS - PROCESO ETL
{'='*80}

Fecha de generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

1. FUENTES DE DATOS PROCESADAS:
   - {', '.join(nombres_unis)}

2. ESTADÍSTICAS GENERALES:
   - Total de registros procesados: {len(df_consolidado):,}
   - Total de columnas: {len(df_consolidado.columns)}
   - Columnas con datos derivados: 5 (PERIODO, NOMBRE_COMPLETO, CATEGORIA_DURACION, FINANCIACION_TOTAL, TIENE_CONVENIO)

3. CALIDAD DE DATOS:
   - Porcentaje promedio de completitud: {completitud['Porcentaje_Completitud'].mean():.2f}%
   - Columnas con 100% completitud: {(completitud['Porcentaje_Completitud'] == 100).sum()}
   - Columnas con <50% completitud: {(completitud['Porcentaje_Completitud'] < 50).sum()}

4. TRANSFORMACIONES APLICADAS:
   ✓ Eliminación de filas vacías
   ✓ Limpieza de espacios en blanco
   ✓ Estandarización de nombres de países
   ✓ Conversión de tipos de datos
   ✓ Eliminación de duplicados
   ✓ Creación de variables derivadas

5. VALIDACIONES PASADAS:
   {chr(10).join(['   ' + v for v in validaciones])}

6. ARCHIVOS GENERADOS:
   - CSV: {archivo_salida}
   - Excel: {archivo_excel}

{'='*80}
DATOS LISTOS PARA ANÁLISIS DESCRIPTIVO Y PREDICTIVO
{'='*80}
"""

# Guardar reporte
archivo_reporte = REPORTS_DIR / 'reporte_calidad_datos.txt'
with open(archivo_reporte, 'w', encoding='utf-8') as f:
    f.write(reporte_calidad)

print(reporte_calidad)
print(f"\n✓ Reporte guardado en: {archivo_reporte}")

---
## 8. VISTA PREVIA DE DATOS LIMPIOS

In [ ]:
print("\nVISTA PREVIA DE DATOS LIMPIOS:")
print("="*80)
display(df_consolidado.head(10))

print("\nINFORMACIÓN DEL DATAFRAME:")
print("="*80)
print(df_consolidado.info())

print("\nESTADÍSTICAS DESCRIPTIVAS:")
print("="*80)
display(df_consolidado.describe())

---
**Fin del Notebook 2**

**Datos procesados y listos para:**
- Análisis descriptivo (Notebook 3)
- Modelos predictivos (Notebook 4)
- Integración con Power BI (Notebook 5)

Continuar con: `03_analisis_descriptivo.ipynb`